# 01 — Làm sạch tài liệu




In [1]:
from pathlib import Path
import re

ROOT = Path("..").resolve()
if not (ROOT / "knowledge").exists():
    ROOT = Path.cwd()
    if (ROOT / "vaxcare-ai" / "knowledge").exists():
        ROOT = ROOT / "vaxcare-ai"

RAW_DIR = ROOT / "knowledge" / "raw"
CLEAN_DIR = ROOT / "knowledge" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT =", ROOT)
print("RAW  =", RAW_DIR)
print("CLEAN=", CLEAN_DIR)


ROOT = D:\TTTN-VaxCare-2026\vaxcare-ai
RAW  = D:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\raw
CLEAN= D:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\clean


In [2]:
def read_text_file(path: Path) -> str:
    raw = path.read_bytes()
    for enc in ("utf-8", "utf-8-sig", "cp1258", "latin-1"):
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            continue
    return raw.decode("utf-8", errors="replace")


def read_pdf(path: Path) -> str:
    try:
        from pypdf import PdfReader
    except ImportError:
        from PyPDF2 import PdfReader  # type: ignore
    reader = PdfReader(str(path))
    parts = []
    for page in reader.pages:
        parts.append(page.extract_text() or "")
    return "\n".join(parts)


def normalize_vietnamese_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\ufeff", "").replace("\u00a0", " ")
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    lines = []
    for line in text.split("\n"):
        s = line.strip()
        if re.fullmatch(r"\d{1,4}", s):
            continue
        if re.fullmatch(r"-{3,}|_{3,}|\*{3,}", s):
            continue
        lines.append(line.rstrip())
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n"


def to_markdown(doc_id: str, body: str, source_name: str) -> str:
    return (
        f"---\n"
        f"doc_id: {doc_id}\n"
        f"source_file: {source_name}\n"
        f"---\n\n"
        f"{body.strip()}\n"
    )


def process_file(path: Path):
    suffix = path.suffix.lower()
    if suffix in {".txt", ".md"}:
        body = read_text_file(path)
    elif suffix == ".pdf":
        body = read_pdf(path)
    else:
        return None
    body = normalize_vietnamese_text(body)
    doc_id = path.stem.lower().replace(" ", "-")
    out_path = CLEAN_DIR / f"{doc_id}.md"
    out_path.write_text(to_markdown(doc_id, body, path.name), encoding="utf-8")
    return out_path


In [3]:
done = 0
for f in sorted(RAW_DIR.rglob("*")):
    if not f.is_file() or f.name.startswith("."):
        continue
    try:
        result = process_file(f)
    except Exception as e:
        print(f"[ERR] {f}: {e}")
        continue
    if result:
        print(f"[OK] {f.relative_to(ROOT)} -> {result.relative_to(ROOT)}")
        done += 1

print(f"Done: {done} file(s)")
print("Files in clean:", list(CLEAN_DIR.glob("*.md")))


[OK] knowledge\raw\byt\6381586785791217441575 QĐ-BYT về việc hướng dẫn khám sàng lọc trước tiêm chủng đối với trẻ em.pdf -> knowledge\clean\6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em.md
[OK] knowledge\raw\byt\Chuan_doan_soi.pdf -> knowledge\clean\chuan_doan_soi.md
[OK] knowledge\raw\byt\luat_phong_chong_benh_truyen_nhiem.pdf -> knowledge\clean\luat_phong_chong_benh_truyen_nhiem.md
[OK] knowledge\raw\byt\ND13-2024.pdf -> knowledge\clean\nd13-2024.md
[OK] knowledge\raw\byt\noi_dung_cua_quyet_dinh_23_ve_an_toan_tiem_chung.pdf -> knowledge\clean\noi_dung_cua_quyet_dinh_23_ve_an_toan_tiem_chung.md
[OK] knowledge\raw\byt\QUYET DINH 3160_QDBYT ngay 10.10.2026 ve viec ban hanh HD Kham sang loc cho nguoi lon va PNCT.pdf -> knowledge\clean\quyet-dinh-3160_qdbyt-ngay-10.10.2026-ve-viec-ban-hanh-hd-kham-sang-loc-cho-nguoi-lon-va-pnct.md
[OK] knowledge\raw\byt\Thong tu 52 ngay 31.12.2025.pdf -> knowledge\clean\thong-tu-52-ngay-31.12.2025.md
[OK] kn